## Notebook 5: Evaluation & Demo

### This Notebook
This notebook demonstrates the complete grant_finder pipeline across four
realistic organization profiles, evaluates system performance, and documents
known limitations and future improvements.

The system answers the question: *"Given a plain-language description of an
organization, which federal grant programs are most likely to fund their work?"*

**Pipeline recap:**
1. Org provides a free-text description + optional type and state
2. Sentence transformer embeds the description into a 384-dim vector
3. FAISS finds the k most semantically similar grant program vectors
4. Llama 3.1 (via Groq) explains fit, flags eligibility concerns, and
   prioritizes the top 3 matches

**Key results:**
- End-to-end query time: ~1.2 seconds
- Semantic matching works beyond keyword overlap (Demo 4)
- System correctly identifies org type eligibility mismatches
- 1,222 federal grant programs indexed from FY2023 USASpending data

**Inputs:** `grant_profiles.csv`, `grant_embeddings.npy`, `grant_index.faiss`

In [1]:
!pip install sentence-transformers faiss-cpu groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/grant_finder'

# Load everything
grant_profiles = pd.read_csv(f'{DRIVE_PATH}/grant_profiles.csv')
grant_embeddings = np.load(f'{DRIVE_PATH}/grant_embeddings.npy')
index = faiss.read_index(f'{DRIVE_PATH}/grant_index.faiss')
st_model = SentenceTransformer('all-MiniLM-L6-v2')
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

print(f'Grant programs loaded: {len(grant_profiles):,}')
print('System ready')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Grant programs loaded: 1,222
System ready


# Pipeline from Notebook 4

In [3]:
def search_grants(query, k=10):
    """Search for matching grant programs given an org description."""
    query_vector = st_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)
    scores, indices = index.search(query_vector, k=k)
    results = grant_profiles.iloc[indices[0]].copy()
    results['similarity_score'] = scores[0]
    return results[[
        'cfda_number', 'cfda_title', 'awarding_agency',
        'num_recipients', 'avg_award', 'min_award', 'max_award',
        'org_types', 'embedding_text', 'similarity_score'
    ]].reset_index(drop=True)


def explain_matches(org_description, search_results, org_type=None, org_state=None):
    """Pass top semantic matches to LLM for explanation and eligibility analysis."""
    grants_context = ""
    for i, row in search_results.head(5).iterrows():
        grants_context += f"""
---
Grant Program {i+1}:
CFDA: {row['cfda_number']}
Title: {row['cfda_title']}
Agency: {row['awarding_agency']}
Avg Award: ${row['avg_award']:,.0f}
Award Range: ${row['min_award']:,.0f} - ${row['max_award']:,.0f}
Typically funds: {row['org_types']}
Similarity score: {row['similarity_score']:.3f}
Program description: {row['embedding_text'][:500]}
"""

    org_context = f"Organization description: {org_description}"
    if org_type:
        org_context += f"\nOrganization type: {org_type}"
    if org_state:
        org_context += f"\nState: {org_state}"

    prompt = f"""You are a federal grant matching assistant. Given an organization's description
and a list of potentially matching federal grant programs, provide a clear analysis of the best matches.

{org_context}

Here are the top semantically matching grant programs from the USASpending database:
{grants_context}

For each grant program, provide:
1. Why it is or isn't a good fit for this organization
2. Any eligibility concerns based on the org type and program description
3. A recommended priority (High / Medium / Low) for pursuing this grant

Important: only reference details that appear in the program description above.
Do not invent specific project names, diseases, or focus areas not mentioned.
Be specific and practical. Focus on the top 3 most promising matches."""

    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content


def find_grants(org_description, org_type=None, org_state=None, k=10):
    """Full pipeline: semantic search + LLM explanation."""
    search_results = search_grants(org_description, k=k)
    explanation = explain_matches(org_description, search_results, org_type, org_state)
    return search_results, explanation

# Demos

In [4]:
# Demo 1: Rural health nonprofit
print('=' * 60)
print('DEMO 1: Rural Health Nonprofit')
print('=' * 60)

results1, explanation1 = find_grants(
    org_description="""We are a nonprofit organization providing primary care,
    mental health counseling, and substance abuse treatment to underserved
    rural communities in Appalachia. We operate three clinics across West
    Virginia serving approximately 8,000 patients annually.""",
    org_type='Nonprofit',
    org_state='WV',
    k=10
)

print('\nTop Semantic Matches:')
print(results1[['cfda_number', 'cfda_title', 'awarding_agency', 'avg_award', 'similarity_score']].to_string())
print('\nLLM Analysis:')
print(explanation1)

DEMO 1: Rural Health Nonprofit

Top Semantic Matches:
   cfda_number                                                                                                     cfda_title                          awarding_agency     avg_award  similarity_score
0       93.647                                                                     SOCIAL SERVICES RESEARCH AND DEMONSTRATION  Department of Health and Human Services  5.030579e+05          0.568637
1       93.859                                                                      BIOMEDICAL RESEARCH AND RESEARCH TRAINING  Department of Health and Human Services  3.502316e+05          0.552178
2       93.788                                                                                                     OPIOID STR  Department of Health and Human Services  4.463162e+05          0.531988
3       93.360  BIOMEDICAL ADVANCED RESEARCH AND DEVELOPMENT AUTHORITY (BARDA), BIODEFENSE MEDICAL COUNTERMEASURE DEVELOPMENT  Department of Health an

In [5]:
# Demo 2: University STEM research
print('=' * 60)
print('DEMO 2: University STEM Research Lab')
print('=' * 60)

results2, explanation2 = find_grants(
    org_description="""We are a public research university running a materials
    science laboratory focused on next-generation battery technology and
    energy storage systems for electric vehicles and grid applications.
    We have 12 PhD researchers and active industry partnerships.""",
    org_type='University / Higher Education',
    org_state='MI',
    k=10
)

print('\nTop Semantic Matches:')
print(results2[['cfda_number', 'cfda_title', 'awarding_agency', 'avg_award', 'similarity_score']].to_string())
print('\nLLM Analysis:')
print(explanation2)

DEMO 2: University STEM Research Lab

Top Semantic Matches:
   cfda_number                                                                                                 cfda_title                          awarding_agency     avg_award  similarity_score
0       81.252                                                                                          ACADEMIC PROGRAMS                     Department of Energy  3.786440e+06          0.532646
1       81.122                                                             ELECTRICITY RESEARCH, DEVELOPMENT AND ANALYSIS                     Department of Energy  4.909870e+05          0.493412
2       81.253                           MANUFACTURING AND ENERGY SUPPLY CHAIN DEMONSTRATIONS AND COMMERCIAL APPLICATIONS                     Department of Energy  1.368675e+08          0.485745
3       11.303                                                                  ECONOMIC DEVELOPMENT TECHNICAL ASSISTANCE                   Department of Commer

In [6]:
# Demo 3: Small business
print('=' * 60)
print('DEMO 3: Small Business — AgTech')
print('=' * 60)

results3, explanation3 = find_grants(
    org_description="""We are a small business developing precision agriculture
    software that uses satellite imagery and machine learning to help farmers
    optimize irrigation, reduce pesticide use, and improve crop yields.
    We currently serve 200 farms across the midwest.""",
    org_type='Small Business',
    org_state='IA',
    k=10
)

print('\nTop Semantic Matches:')
print(results3[['cfda_number', 'cfda_title', 'awarding_agency', 'avg_award', 'similarity_score']].to_string())
print('\nLLM Analysis:')
print(explanation3)

DEMO 3: Small Business — AgTech

Top Semantic Matches:
   cfda_number                                                                           cfda_title            awarding_agency     avg_award  similarity_score
0       10.523                                           CENTERS OF EXCELLENCE AT 1890 INSTITUTIONS  Department of Agriculture  1.374729e+06          0.557959
1       10.230                                                                   FARM OF THE FUTURE  Department of Agriculture  2.342184e+06          0.556586
2       10.533                                                                      SNAP-ED TOOLKIT  Department of Agriculture  2.357640e+05          0.532033
3       10.329                                                    MINOR CROP PEST MANAGEMENT (IR-4)  Department of Agriculture  1.765572e+05          0.510371
4       10.308  RESIDENT INSTRUCTION, AGRICULTURE, AND FOOD SCIENCE FACILITIES AND EQUIPMENT GRANTS  Department of Agriculture  2.497879e+05          

In [7]:
# Demo 4: Something niche to stress test the system
print('=' * 60)
print('DEMO 4: Niche Case — Native Language Preservation')
print('=' * 60)

results4, explanation4 = find_grants(
    org_description="""We are a nonprofit dedicated to preserving and revitalizing
    endangered Native American languages through digital archiving, community
    education programs, and intergenerational language transmission projects
    in partnership with tribal communities in the Pacific Northwest.""",
    org_type='Nonprofit',
    org_state='WA',
    k=10
)

print('\nTop Semantic Matches:')
print(results4[['cfda_number', 'cfda_title', 'awarding_agency', 'avg_award', 'similarity_score']].to_string())
print('\nLLM Analysis:')
print(explanation4)

DEMO 4: Niche Case — Native Language Preservation

Top Semantic Matches:
   cfda_number                                                                 cfda_title                          awarding_agency     avg_award  similarity_score
0       93.587  PROMOTE THE SURVIVAL AND CONTINUING VITALITY OF NATIVE AMERICAN LANGUAGES  Department of Health and Human Services  2.767548e+05          0.600675
1       21.012                                                         NATIVE INITIATIVES               Department of the Treasury  1.056467e+06          0.560329
2       15.071                                PACIFIC NORTHWEST AND HAWAIIAN ISLANDS ARTS               Department of the Interior  7.467287e+05          0.556849
3       84.365                                  ENGLISH LANGUAGE ACQUISITION STATE GRANTS                  Department of Education  5.433016e+05          0.545878
4       84.299                   INDIAN EDUCATION -- SPECIAL PROGRAMS FOR INDIAN CHILDREN                  Depar

# Evaluation summary

STRENGTHS:
1. Semantic matching works beyond keywords
   - Native language preservation query correctly surfaced exact program match
     with no keyword overlap
   - Cancer detection query found relevant NIH programs without needing
     specific medical terminology

2. Org type awareness
   - LLM correctly flags when programs typically fund universities vs nonprofits
     vs small businesses
   - AgTech small business results correctly noted most agriculture grants
     favor academic institutions

3. Geographic relevance
   - Rural Appalachia context influenced opioid and rural health rankings
   - Pacific Northwest context surfaced in Native language results

LIMITATIONS:
1. 256 token truncation
   - Embedding model truncates all text at 256 tokens (~200 words)
   - Large programs with many recipients may lose important distinguishing
     information in later descriptions

2. FY2023 only
   - Some grant programs only award every 2-3 years
   - An org searching today may miss programs that last awarded in 2021-2022
   - Fix: add FY2021 and FY2022 data to grant profiles

3. No live eligibility data
   - System cannot check current open/closed status of grant programs
   - Fix: integrate Grants.gov API for live opportunity status

4. LLM hallucination risk
   - Smaller models (Llama 3.1 8B) occasionally invent specific details
     not present in the program description
   - Fix: upgrade to larger model or add stricter prompt constraints

5. Federal grants only
   - Does not cover state grants, foundation grants, or corporate grants
   - Significant funding sources remain outside scope

DATASET STATS:
- 275,597 award records processed
- 23,111 unique organizations
- 1,222 grant programs indexed
- 3 org types covered: Nonprofits, Universities, Small Businesses
- Source: USASpending.gov FY2023 Financial Assistance Archive
"""

In [8]:
# Final timing test - how fast is a query end to end?
import time

start = time.time()
results, explanation = find_grants(
    org_description="nonprofit providing food assistance and hunger relief programs in urban communities",
    org_type='Nonprofit',
    org_state='NY',
    k=10
)
end = time.time()

print(f'End-to-end query time: {end - start:.2f} seconds')
print(f'  - Semantic search: near instant (FAISS in-memory)')
print(f'  - LLM explanation: most of the latency (Groq API call)')

End-to-end query time: 1.24 seconds
  - Semantic search: near instant (FAISS in-memory)
  - LLM explanation: most of the latency (Groq API call)
